In [10]:
from pathlib import Path
import sys

# Find the DocuMind project directory
project_dir = Path.cwd()

# If the notebook is running inside notebooks/, go up one level
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

backend_dir = project_dir / "backend"

# Make the backend directory visible to Python
sys.path.insert(0, str(backend_dir))

print("Project directory:", project_dir)
print("Backend directory:", backend_dir)
print("Backend exists:", backend_dir.exists())

Project directory: /run/media/biplob/e1a9a613-7e49-4b9e-81ed-a1c7ad2b4cae/biplob/DocuMind
Backend directory: /run/media/biplob/e1a9a613-7e49-4b9e-81ed-a1c7ad2b4cae/biplob/DocuMind/backend
Backend exists: True


In [11]:
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.


In [12]:
from app.services.text_splitter import split_text, search_chunks
from app.services.document_loader import extract_text_from_pdf

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All imports successful!")

All imports successful!


In [13]:
text = extract_text_from_pdf(Path("../data/sample.pdf").read_bytes())
chunks = split_text(text, chunk_size=500, overlap=50)
print("Characters:", len(text), "| chunks:", len(chunks))
print(chunks[0][:300])

Characters: 2755 | chunks: 7
Page 1
Python is a programming language known for readable syntax. Python is a programming
language known for readable syntax. Python is a programming language known for readable
syntax. Python is a programming language known for readable syntax. Python is a
programming language known for readable s


In [14]:
def show(query, chunks, top_k=3,**opts):
    results = search_chunks(query, chunks, top_k,**opts)
    print(f"\nQuestion: {query}")
    if not results:
        print(" (no match)")
    for r in results:
        preview = r["text"][:150].replace("\n", " ")
        print(f" chunk{r['chunk_index']:>3} | score {r['score']:.3f} | {preview}...")


show("Python is a programming language", chunks)


Question: Python is a programming language
 chunk  0 | score 0.673 | Page 1 Python is a programming language known for readable syntax. Python is a programming language known for readable syntax. Python is a programming...
 chunk  1 | score 0.633 | language known for readable syntax. Python is a programming language known for readable syntax. Python is a programming language known for readable sy...
 chunk  4 | score 0.173 | in tables. PostgreSQL is a relational database that stores data in tables.   Page 3 FastAPI is a web framework for building APIs with Python. FastAPI ...


In [15]:
tests = [
    ("question 1", "phrase from the right chunk"),
    ("question 2", "phrase from the right chunk"),
    ("question 3", "phrase from the right chunk"),
    ("question 4", "phrase from the right chunk"),
    ("question 5", "phrase from the right chunk"),
]


def rank(query, chunks, **opts):
    vec = TfidfVectorizer(**opts)
    matrix = vec.fit_transform(chunks)
    scores = cosine_similarity(vec.transform([query]), matrix).flatten()
    return scores.argsort()[::-1], scores


def evaluate(chunks, tests, top_k=3, verbose=False, **opts):
    top1 = found = 0
    for question, expected in tests:
        order, scores = rank(question, chunks, **opts)
        position = None
        for place, idx in enumerate(order[:top_k], start=1):
            if expected.lower() in chunks[idx].lower():
                position = place
                break
        top1 += position == 1
        found += position is not None
        if verbose:
            print(
                f"{position or 'miss':>5} | best score {scores[order[0]]:.3f} | {question}"
            )

    return top1, found


top1, found = evaluate(chunks, tests, verbose=True)
print(
    f"\ncorrect chunk first: {top1}/{len(tests)} | within top 3: {found}/{len(tests)}"
)

 miss | best score 0.000 | question 1
 miss | best score 0.000 | question 2
 miss | best score 0.000 | question 3
 miss | best score 0.000 | question 4
 miss | best score 0.000 | question 5

correct chunk first: 0/5 | within top 3: 0/5


In [16]:
for size, overlap in [(300, 30), (500, 50), (800, 80), (1200, 120)]:
    c = split_text(text, size, overlap)
    top1, found = evaluate(c, tests)
    print(
        f"size {size:>4} overlap {overlap:>3} | chunks {len(c):>3} | first {top1}/{len(tests)} | top3 {found}/{len(tests)}"
    )

size  300 overlap  30 | chunks  11 | first 0/5 | top3 0/5
size  500 overlap  50 | chunks   7 | first 0/5 | top3 0/5
size  800 overlap  80 | chunks   4 | first 0/5 | top3 0/5
size 1200 overlap 120 | chunks   3 | first 0/5 | top3 0/5


In [17]:
settings = {
    "default": {},
    "stop words": {"stop_words": "english"},
    "bigrams": {"ngram_range": (1, 2)},
    "sublinear tf": {"sublinear_tf": True},
    "all three": {"stop_words": "english", "ngram_range": (1, 2), "sublinear_tf": True},
}

for name, opts in settings.items():
    top1, found = evaluate(chunks, tests, **opts)
    print(f"{name:<14} | first {top1}/{len(tests)} | top3 {found}/{len(tests)}")

default        | first 0/5 | top3 0/5
stop words     | first 0/5 | top3 0/5
bigrams        | first 0/5 | top3 0/5
sublinear tf   | first 0/5 | top3 0/5
all three      | first 0/5 | top3 0/5


In [18]:
show("What is the capital of France?", chunks)
show("What is the capital of France?", chunks, stop_words="english")


Question: What is the capital of France?
 chunk  4 | score 0.229 | in tables. PostgreSQL is a relational database that stores data in tables.   Page 3 FastAPI is a web framework for building APIs with Python. FastAPI ...
 chunk  2 | score 0.228 | yntax.   Page 2 PostgreSQL is a relational database that stores data in tables. PostgreSQL is a relational database that stores data in tables. Postgr...
 chunk  0 | score 0.227 | Page 1 Python is a programming language known for readable syntax. Python is a programming language known for readable syntax. Python is a programming...

Question: What is the capital of France?
 (no match)
